In [1]:
import os
import gc
import time
import pickle
import joblib 
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

def generar_explicabilidad_shap_mortalidad_rf():
    """
    Pipeline unificado de explicabilidad SHAP para MORTALIDAD (Random Forest).
    Calcula: SHAP crudo, Porcentajes, Paneles, Direcciones Numéricas/Categóricas 
    para la clase 1 (Fallecido) y reporte de diferencias Global vs Oncológico.
    """
    target_name = 'MORTALIDAD'
    idx_clase_alta = 1  # Clase 1: Fallecido
    nombre_efecto_str = 'Mortalidad (Clase 1)'

    # -------------------------------------------------------------------------
    # CONFIGURACIÓN DE RUTAS Y PARÁMETROS
    # -------------------------------------------------------------------------
    dir_datos = "../../Datos/Datasets Finales"
    dir_modelos = "../../Resultados/Resultados (etapa 3 y 4)/Random_Forest"
    dir_base_resultados = f"../../Resultados/Resultados (etapa 5)/SHAP_{target_name}"
    os.makedirs(dir_base_resultados, exist_ok=True)
    
    nombre_modelo = f"Modelo_Optimo_RF_{target_name}.pkl"
    ruta_modelo = os.path.join(dir_modelos, nombre_modelo)
    
    cols_excluir = ['CONSUMO_RECURSOS', 'SEVERIDAD', 'MORTALIDAD', 'CATEGORIA_CANCER']
    vars_num = ['CANTIDAD_TRASLADOS', 'CARGA_ONCOLOGICA', 'DIAS_ESTADIA', 'EDAD', 'NUM_COMORBILIDADES', 'NUM_PROCEDIMIENTOS']
    
    print("="*80)
    print(f"INICIANDO FASE 5: SHAP BINARIO - TARGET: {target_name}")
    print(f"Foco clínico de análisis direccional: {nombre_efecto_str}")
    print(f"Hora de inicio: {time.strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*80)
    
    if not os.path.exists(ruta_modelo):
        print(f"ERROR: No se encontró el modelo en la ruta: {ruta_modelo}")
        return
        
    print(f"-> Cargando modelo óptimo pre-entrenado ({nombre_modelo})...")
    modelo_rf = joblib.load(ruta_modelo)
        
    if hasattr(modelo_rf, 'best_estimator_'): modelo_rf = modelo_rf.best_estimator_
    if hasattr(modelo_rf, 'steps'): modelo_rf = modelo_rf.steps[-1][1]
    if hasattr(modelo_rf, 'calibrated_classifiers_'): modelo_rf = modelo_rf.calibrated_classifiers_[0].estimator

    if hasattr(modelo_rf, 'feature_names_in_'):
        features = modelo_rf.feature_names_in_.tolist()
    else:
        df_dummy = pd.read_csv(os.path.join(dir_datos, "dataset_prueba_onco.csv"), nrows=1)
        features = [c for c in df_dummy.columns if c not in cols_excluir]

    print("-> Inicializando SHAP TreeExplainer nativo...")
    explainer = shap.TreeExplainer(modelo_rf)
    
    # -------------------------------------------------------------------------
    # FUNCIÓN INTERNA DE PROCESAMIENTO
    # -------------------------------------------------------------------------
    def procesar_enfoque_shap_binario(df_origen, tipo_enfoque, nombre_carpeta_sub):
        print(f"\n--- Procesando enfoque: {tipo_enfoque.upper()} (Datos: {len(df_origen)}) ---")
        
        dir_sub_enfoque = os.path.join(dir_base_resultados, nombre_carpeta_sub)
        dir_dependence = os.path.join(dir_sub_enfoque, "Dependence_Plots")
        os.makedirs(dir_dependence, exist_ok=True)
        
        X_shap = df_origen[features].astype('float32')
        del df_origen; gc.collect()
        
        inicio_time = time.time()
        
        # Bloques de 500 para Random Forest (Evita Swap Thrashing)
        batch_size = 500
        resultados_list = []
        n_batches = (len(X_shap) // batch_size) + (1 if len(X_shap) % batch_size != 0 else 0)
        
        for i in range(0, len(X_shap), batch_size):
            batch = X_shap.iloc[i:i+batch_size]
            if (i // batch_size + 1) % 10 == 0 or (i // batch_size + 1) == 1:
                print(f"      -> Procesando bloque {i//batch_size + 1} de {n_batches}...")
            
            shap_output = explainer.shap_values(batch, check_additivity=False, approximate=True)
            
            # Reestructurar la salida a 3D (Muestras, Variables, Clases) para compatibilidad
            if isinstance(shap_output, list):
                shap_mat_batch = np.stack(shap_output, axis=2)
            elif len(shap_output.shape) == 3:
                shap_mat_batch = shap_output
            else:
                shap_mat_batch = np.stack([shap_output * -1, shap_output], axis=2)
                
            resultados_list.append(shap_mat_batch)
            del batch, shap_output, shap_mat_batch; gc.collect()
            
        matriz_shap = np.concatenate(resultados_list, axis=0)
        print(f"   -> SHAP completado en {round((time.time() - inicio_time)/60, 2)} minutos.")
        
        # --- FILTRO ONCOLÓGICO DE CONSTANTES ---
        if "onco" in nombre_carpeta_sub.lower():
            varianzas = X_shap.var()
            cols_a_eliminar = varianzas[varianzas == 0].index.tolist()
            if 'CATEGORIA_CANCER_SIN_CANCER' in X_shap.columns and 'CATEGORIA_CANCER_SIN_CANCER' not in cols_a_eliminar:
                cols_a_eliminar.append('CATEGORIA_CANCER_SIN_CANCER')
                
            if cols_a_eliminar:
                idx_a_eliminar = [X_shap.columns.get_loc(col) for col in cols_a_eliminar]
                X_shap = X_shap.drop(columns=cols_a_eliminar)
                matriz_shap = np.delete(matriz_shap, idx_a_eliminar, axis=1)
                print(f"      FILTRO: Se excluyeron {len(cols_a_eliminar)} variables constantes.")
        
        n_clases = matriz_shap.shape[2]
        sufijo_archivo = "GLOBAL" if "global" in nombre_carpeta_sub.lower() else "ONCO"
        
        # 1. Guardar respaldo (.npy)
        ruta_npy = os.path.join(dir_sub_enfoque, f"BACKUP_MATRIZ_GLOBAL_{target_name}_{sufijo_archivo}.npy")
        np.save(ruta_npy, matriz_shap)
        
        # 2. Exportar CSV Numérico Absoluto
        shap_abs = np.abs(matriz_shap).mean(axis=0) 
        impacto_total = shap_abs.sum(axis=1)
        columnas_csv = [f"Clase_{i}" for i in range(n_clases)]
        df_shap_imp = pd.DataFrame(shap_abs, index=X_shap.columns, columns=columnas_csv)
        df_shap_imp['Impacto_Total'] = impacto_total
        
        df_shap_imp = df_shap_imp.sort_values(by='Impacto_Total', ascending=False)
        ruta_csv = os.path.join(dir_sub_enfoque, f"SHAP_Valores_{target_name}_{sufijo_archivo}.csv")
        df_shap_imp.to_csv(ruta_csv, index_label='Variable')

        # 3. Exportar CSV Porcentual
        print("   -> Generando matriz de importancias porcentuales...")
        df_shap_porcentajes = df_shap_imp.copy()
        cols_num_pct = df_shap_porcentajes.select_dtypes(include=['number']).columns
        for col in cols_num_pct:
            suma_total = df_shap_porcentajes[col].sum()
            if suma_total > 0:
                df_shap_porcentajes[col] = (df_shap_porcentajes[col] / suma_total) * 100
        
        ruta_csv_pct = os.path.join(dir_sub_enfoque, f"SHAP_Valores_{target_name}_{sufijo_archivo}_PORCENTAJES.csv")
        df_shap_porcentajes.to_csv(ruta_csv_pct, index_label='Variable')
        
        df_retorno = df_shap_porcentajes.reset_index().rename(columns={'index': 'Variable'})
        
        # 4. Gráficos Summary Plots
        plt.figure(figsize=(12, 8))
        df_top20 = df_shap_imp.head(20).drop(columns=['Impacto_Total']).iloc[::-1]
        df_top20.plot(kind='barh', stacked=True, figsize=(12, 8), cmap='bwr', ax=plt.gca())
        plt.title(f'Top 20 Variables SHAP - Enfoque {tipo_enfoque} ({target_name})', fontsize=14)
        plt.xlabel('Valor SHAP absoluto promedio')
        plt.tight_layout()
        plt.savefig(os.path.join(dir_sub_enfoque, f"SHAP_Summary_General_{target_name}_{sufijo_archivo}.png"), dpi=300)
        plt.close()
        
        plt.figure(figsize=(12, 8))
        vars_cat_ohe = [col for col in df_shap_imp.index if col not in vars_num]
        df_top20_cat = df_shap_imp.loc[vars_cat_ohe].head(20).drop(columns=['Impacto_Total']).iloc[::-1]
        df_top20_cat.plot(kind='barh', stacked=True, figsize=(12, 8), cmap='coolwarm', ax=plt.gca())
        plt.title(f'Top 20 Variables Categóricas SHAP - Enfoque {tipo_enfoque} ({target_name})', fontsize=14)
        plt.xlabel('Valor SHAP absoluto promedio')
        plt.tight_layout()
        plt.savefig(os.path.join(dir_sub_enfoque, f"SHAP_Summary_Categoricas_{target_name}_{sufijo_archivo}.png"), dpi=300)
        plt.close()

        # Aislar matriz de la clase 1 (Fallecido)
        matriz_clase_alta = matriz_shap[:, :, idx_clase_alta]
        
        # 5. Análisis Direccional: Variables Numéricas (Cuartiles)
        print(f"   -> Extrayendo impacto direccional ({nombre_efecto_str}) para variables numéricas...")
        rangos_direccionales = []
        for v_num in vars_num:
            if v_num in X_shap.columns:
                try: bins_serie = pd.qcut(X_shap[v_num], q=4, duplicates='drop')
                except: bins_serie = pd.cut(X_shap[v_num], bins=4)
                
                idx_var = X_shap.columns.get_loc(v_num)
                for rango in bins_serie.cat.categories:
                    indices_rango = (bins_serie == rango)
                    n_pacientes = indices_rango.sum()
                    if n_pacientes > 0:
                        promedio_crudo = matriz_clase_alta[indices_rango, idx_var].mean()
                        if promedio_crudo > 0: efecto = "Aumenta Mortalidad (+)"
                        elif promedio_crudo < 0: efecto = "Protector / Supervivencia (-)"
                        else: efecto = "Neutral"

                        rangos_direccionales.append({
                            "Variable": v_num, "Rango": str(rango), "N_Pacientes": n_pacientes,
                            f"SHAP_Crudo_{target_name}_Clase{idx_clase_alta}": promedio_crudo, "Efecto_Clinico": efecto
                        })
                        
        df_rangos_num = pd.DataFrame(rangos_direccionales)
        df_rangos_num.to_csv(os.path.join(dir_sub_enfoque, f"Rangos_Direccion_Numericas_{target_name}_{sufijo_archivo}.csv"), index=False)

        # 6. Análisis Direccional: Variables Categóricas (OHE 0 vs 1)
        print(f"   -> Extrayendo impacto direccional ({nombre_efecto_str}) para variables categóricas (OHE)...")
        cat_direccionales = []
        for v_cat in vars_cat_ohe:
            idx_var = X_shap.columns.get_loc(v_cat)
            for valor_cat in [0, 1]:
                indices_cat = (X_shap[v_cat] == valor_cat)
                n_pacientes_cat = indices_cat.sum()
                
                if n_pacientes_cat > 0:
                    promedio_crudo = matriz_clase_alta[indices_cat, idx_var].mean()
                    if promedio_crudo > 0: efecto = "Aumenta Mortalidad (+)"
                    elif promedio_crudo < 0: efecto = "Protector / Supervivencia (-)"
                    else: efecto = "Neutral"

                    cat_direccionales.append({
                        "Variable": v_cat,
                        "Condicion_OHE": valor_cat,
                        "Significado": "Presencia (1)" if valor_cat == 1 else "Ausencia (0)",
                        "N_Pacientes": n_pacientes_cat,
                        f"SHAP_Crudo_{target_name}_Clase{idx_clase_alta}": promedio_crudo,
                        "Efecto_Clinico": efecto
                    })
                    
        df_cat_direccional = pd.DataFrame(cat_direccionales)
        df_cat_direccional.to_csv(os.path.join(dir_sub_enfoque, f"Rangos_Direccion_Categoricas_{target_name}_{sufijo_archivo}.csv"), index=False)

        # 7. Paneles de Dependencia (Top 20)
        print(f"   -> Generando paneles de dependencia en carpeta...")
        top_20_vars = df_shap_imp.head(20).index.tolist()
        for var in top_20_vars:
            if var in X_shap.columns:
                fig, ax = plt.subplots(figsize=(6, 4.5))
                valores_sh_fallecido = matriz_shap[:, :, 1]
                shap.dependence_plot(var, valores_sh_fallecido, X_shap, interaction_index=None, ax=ax, show=False)
                ax.set_title('Impacto en Riesgo de Fallecimiento (Clase 1)', fontsize=10)
                fig.suptitle(f'Dependence Plot: {var} ({sufijo_archivo})', fontsize=11, y=1.02)
                plt.tight_layout()
                plt.savefig(os.path.join(dir_dependence, f"SHAP_Dependence_{var}.png"), dpi=200, bbox_inches='tight')
                plt.close()
                
        print(f"   Liberando memoria asignada al enfoque {tipo_enfoque}...")
        del X_shap, matriz_shap, df_shap_imp, df_shap_porcentajes
        gc.collect()
        
        return df_retorno

    # -------------------------------------------------------------------------
    # EJECUCIÓN SECUENCIAL Y REPORTE CRUZADO (DIFERENCIAS)
    # -------------------------------------------------------------------------
    
    print("\n--- PASO A: Cargando datos para análisis global ---")
    df_onco_test = pd.read_csv(os.path.join(dir_datos, "dataset_prueba_onco.csv"), low_memory=False)
    df_control_test = pd.read_csv(os.path.join(dir_datos, "dataset_prueba_control.csv"), low_memory=False)
    
    n_onco_total = len(df_onco_test)
    n_control_needed = 200000 - n_onco_total 
    
    proporciones_control = df_control_test[target_name].value_counts(normalize=True)
    df_ctrl_sample = df_control_test.groupby(target_name, group_keys=False).apply(
        lambda x: x.sample(min(len(x), int(np.round(n_control_needed * proporciones_control[x.name]))), random_state=42)
    )
    
    if len(df_ctrl_sample) != n_control_needed:
        if len(df_ctrl_sample) < n_control_needed:
            dif = n_control_needed - len(df_ctrl_sample)
            extras = df_control_test.drop(df_ctrl_sample.index).sample(n=dif, random_state=42)
            df_ctrl_sample = pd.concat([df_ctrl_sample, extras])
        elif len(df_ctrl_sample) > n_control_needed:
            dif = len(df_ctrl_sample) - n_control_needed
            df_ctrl_sample = df_ctrl_sample.drop(df_ctrl_sample.sample(n=dif, random_state=42).index)
            
    df_test_global = pd.concat([df_onco_test, df_ctrl_sample], ignore_index=True).sample(frac=1, random_state=42)
    del df_control_test, df_ctrl_sample; gc.collect() 
    
    df_global_pct = procesar_enfoque_shap_binario(df_test_global, "Global (Onco + Control)", "Valores SHAP (global)")
    
    print("\n--- PASO B: Cargando datos para análisis oncológico ---")
    df_onco_pct = procesar_enfoque_shap_binario(df_onco_test, "Oncológico Estricto", "Valores SHAP (oncologicos)")
    
    print("\n--- PASO C: Generando reporte comparativo (Diferencias Top 20 Onco vs Global) ---")
    df_global_pct['Posicion_Global'] = df_global_pct.index + 1
    df_onco_pct['Posicion_Onco'] = df_onco_pct.index + 1
    
    top_20_onco = df_onco_pct.head(20).copy()
    
    df_comparacion = pd.merge(top_20_onco, df_global_pct, on='Variable', suffixes=('_Onco', '_Global'), how='left')
    df_comparacion['Diferencia_Impacto_Total'] = df_comparacion['Impacto_Total_Onco'] - df_comparacion['Impacto_Total_Global']
    
    col_clase = f"Clase_{idx_clase_alta}"
    col_clase_onco = f"{col_clase}_Onco"
    col_clase_global = f"{col_clase}_Global"
    
    columnas_finales = ['Variable', 'Posicion_Onco', 'Posicion_Global', 'Impacto_Total_Onco', 'Impacto_Total_Global', 'Diferencia_Impacto_Total']
    
    if col_clase_onco in df_comparacion.columns and col_clase_global in df_comparacion.columns:
        nombre_diferencia_clase = f"Diferencia_{col_clase}"
        df_comparacion[nombre_diferencia_clase] = df_comparacion[col_clase_onco] - df_comparacion[col_clase_global]
        columnas_finales.extend([col_clase_onco, col_clase_global, nombre_diferencia_clase])
        
    df_final = df_comparacion[columnas_finales]
    ruta_diferencias = os.path.join(dir_base_resultados, f"Diferencias_SHAP_{target_name}_ONCO_GLOBAL.csv")
    df_final.to_csv(ruta_diferencias, index=False)
    
    print("\n" + "="*80)
    print("PROCESO UNIFICADO FINALIZADO CON ÉXITO")
    print(f"Reportes guardados en: {dir_base_resultados}")
    print("="*80)

In [2]:
# Ejecutar el pipeline de forma directa
generar_explicabilidad_shap_mortalidad_rf()

INICIANDO FASE 5: SHAP BINARIO - TARGET: MORTALIDAD
Foco clínico de análisis direccional: Mortalidad (Clase 1)
Hora de inicio: 2026-07-16 03:05:58
-> Cargando modelo óptimo pre-entrenado (Modelo_Optimo_RF_MORTALIDAD.pkl)...
-> Inicializando SHAP TreeExplainer nativo...

--- PASO A: Cargando datos para análisis global ---

--- Procesando enfoque: GLOBAL (ONCO + CONTROL) (Datos: 200000) ---
      -> Procesando bloque 1 de 400...
      -> Procesando bloque 10 de 400...
      -> Procesando bloque 20 de 400...
      -> Procesando bloque 30 de 400...
      -> Procesando bloque 40 de 400...
      -> Procesando bloque 50 de 400...
      -> Procesando bloque 60 de 400...
      -> Procesando bloque 70 de 400...
      -> Procesando bloque 80 de 400...
      -> Procesando bloque 90 de 400...
      -> Procesando bloque 100 de 400...
      -> Procesando bloque 110 de 400...
      -> Procesando bloque 120 de 400...
      -> Procesando bloque 130 de 400...
      -> Procesando bloque 140 de 400...
    

Dependence plots de tipos de cáncer

In [2]:
import os
import gc
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt


def generar_dependence_plots_tipos_cancer():

    # ======================================================================
    # RUTAS
    # ======================================================================

    dir_datos = "../../Datos/Datasets Finales"

    dir_shap = (
        "../../Resultados/Resultados (etapa 5)/"
        "SHAP_MORTALIDAD/Valores SHAP (oncologicos)"
    )

    ruta_backup = os.path.join(
        dir_shap,
        "BACKUP_MATRIZ_GLOBAL_MORTALIDAD_ONCO.npy"
    )

    ruta_dataset = os.path.join(
        dir_datos,
        "dataset_prueba_onco.csv"
    )

    dir_salida = os.path.join(
        dir_shap,
        "Dependence_Plots",
        "Dependence_Plots_TIPOS_CANCER"
    )

    os.makedirs(dir_salida, exist_ok=True)

    # ======================================================================
    # VARIABLES DE CÁNCER
    # ======================================================================

    variables_cancer = [

        "CATEGORIA_CANCER_C15_C26",
        "CATEGORIA_CANCER_C30_C39",
        "CATEGORIA_CANCER_C40_C41",
        "CATEGORIA_CANCER_C43_C44",
        "CATEGORIA_CANCER_C45_C49",
        "CATEGORIA_CANCER_C50",
        "CATEGORIA_CANCER_C51_C58",
        "CATEGORIA_CANCER_C60_C63",
        "CATEGORIA_CANCER_C64_C68",
        "CATEGORIA_CANCER_C69_C72",
        "CATEGORIA_CANCER_C73_C75",
        "CATEGORIA_CANCER_C76_C80",
        "CATEGORIA_CANCER_C81_C96",
        "CATEGORIA_CANCER_C97"

    ]

    # ======================================================================
    # CARGAR DATASET
    # ======================================================================

    print("Cargando dataset oncológico...")

    df = pd.read_csv(
        ruta_dataset,
        low_memory=False
    )

    # Eliminar únicamente columnas objetivo que existan
    cols_excluir = [
        "MORTALIDAD",
        "SEVERIDAD",
        "CONSUMO_RECURSOS",
        "CATEGORIA_CANCER"
    ]

    cols_excluir = [
        c for c in cols_excluir
        if c in df.columns
    ]

    X_shap = df.drop(
        columns=cols_excluir,
        errors="ignore"
    ).astype("float32")

    del df
    gc.collect()

    # ======================================================================
    # REPRODUCIR EL FILTRO DEL PIPELINE ORIGINAL
    # ======================================================================

    print("Aplicando filtro de variables constantes...")

    varianzas = X_shap.var()

    cols_a_eliminar = varianzas[
        varianzas == 0
    ].index.tolist()

    if (
        "CATEGORIA_CANCER_SIN_CANCER" in X_shap.columns
        and
        "CATEGORIA_CANCER_SIN_CANCER" not in cols_a_eliminar
    ):
        cols_a_eliminar.append(
            "CATEGORIA_CANCER_SIN_CANCER"
        )

    if len(cols_a_eliminar):

        print(f"Se eliminaron {len(cols_a_eliminar)} variables constantes.")

        X_shap = X_shap.drop(
            columns=cols_a_eliminar
        )

    # ======================================================================
    # CARGAR MATRIZ SHAP
    # ======================================================================

    print("Cargando backup SHAP...")

    matriz_shap = np.load(ruta_backup)

    print(f"Matriz SHAP : {matriz_shap.shape}")
    print(f"Dataset      : {X_shap.shape}")

    if matriz_shap.shape[1] != X_shap.shape[1]:

        print("\nERROR")
        print("Las dimensiones no coinciden.")
        print(f"Matriz SHAP : {matriz_shap.shape[1]} variables")
        print(f"Dataset     : {X_shap.shape[1]} variables")

        # Mostrar diferencias para depuración

        print("\nPrimeras variables del dataset:")

        print(X_shap.columns.tolist()[:15])

        raise ValueError(
            "La matriz SHAP y el dataset no tienen el mismo número de variables."
        )

    # Clase 1 = Fallecido

    valores_shap = matriz_shap[:, :, 1]

    # ======================================================================
    # GENERAR DEPENDENCE PLOTS
    # ======================================================================

    print("\nGenerando dependence plots...\n")

    generados = 0

    for variable in variables_cancer:

        if variable not in X_shap.columns:

            print(f"{variable} no existe.")
            continue

        print(f"Generando: {variable}")

        fig, ax = plt.subplots(figsize=(6, 4.5))

        shap.dependence_plot(
            variable,
            valores_shap,
            X_shap,
            interaction_index=None,
            ax=ax,
            show=False
        )

        ax.set_title(
            "Impacto en Riesgo de Fallecimiento (Clase 1)",
            fontsize=10
        )

        fig.suptitle(
            f"Dependence Plot: {variable}",
            fontsize=11,
            y=1.02
        )

        plt.tight_layout()

        plt.savefig(
            os.path.join(
                dir_salida,
                f"SHAP_Dependence_{variable}.png"
            ),
            dpi=250,
            bbox_inches="tight"
        )

        plt.close()

        generados += 1

    del matriz_shap
    del X_shap

    gc.collect()

    print("\n==============================================")
    print("Proceso terminado.")
    print(f"Dependence plots generados: {generados}")
    print(f"Carpeta de salida:\n{dir_salida}")
    print("==============================================")


# ==========================================================================
# EJECUTAR
# ==========================================================================

generar_dependence_plots_tipos_cancer()

Cargando dataset oncológico...
Aplicando filtro de variables constantes...
Se eliminaron 3 variables constantes.
Cargando backup SHAP...
Matriz SHAP : (97552, 107, 2)
Dataset      : (97552, 107)

Generando dependence plots...

Generando: CATEGORIA_CANCER_C15_C26
Generando: CATEGORIA_CANCER_C30_C39
Generando: CATEGORIA_CANCER_C40_C41
Generando: CATEGORIA_CANCER_C43_C44
Generando: CATEGORIA_CANCER_C45_C49
Generando: CATEGORIA_CANCER_C50
Generando: CATEGORIA_CANCER_C51_C58
Generando: CATEGORIA_CANCER_C60_C63
Generando: CATEGORIA_CANCER_C64_C68
Generando: CATEGORIA_CANCER_C69_C72
Generando: CATEGORIA_CANCER_C73_C75
Generando: CATEGORIA_CANCER_C76_C80
Generando: CATEGORIA_CANCER_C81_C96
Generando: CATEGORIA_CANCER_C97

Proceso terminado.
Dependence plots generados: 14
Carpeta de salida:
../../Resultados/Resultados (etapa 5)/SHAP_MORTALIDAD/Valores SHAP (oncologicos)\Dependence_Plots\Dependence_Plots_TIPOS_CANCER


Dependence plots de shap global de variables que están dentro del top 20 de variables oncológicas y fuera del top en el shap global

In [1]:
import os
import gc
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt

def generar_dependence_plots_globales():

    # ======================================================================
    # RUTAS
    # ======================================================================

    dir_datos = "../../Datos/Datasets Finales"

    dir_shap = (
        "../../Resultados/Resultados (etapa 5)/"
        "SHAP_MORTALIDAD/Valores SHAP (global)"
    )

    ruta_backup = os.path.join(
        dir_shap,
        "BACKUP_MATRIZ_GLOBAL_MORTALIDAD_GLOBAL.npy"
    )

    dir_salida = os.path.join(
        dir_shap,
        "Dependence_Plots"
    )

    os.makedirs(dir_salida, exist_ok=True)

    # ======================================================================
    # VARIABLES ESPECÍFICAS
    # ======================================================================

    variables_objetivo = [
        "TIPO_DIAGNOSTICO_ONCO_SECUNDARIO",
        "COMORBILIDAD_PRINCIPAL_SIN_COMORBILIDAD",
        "TIPO_PROCEDIMIENTO_SISTEMA_DIGESTIVO"
    ]

    # ======================================================================
    # CARGAR Y RECONSTRUIR DATASET GLOBAL (200.000 registros)
    # ======================================================================

    print("Reconstruyendo dataset global (Onco + Control)...")
    
    target_name = 'MORTALIDAD'
    
    df_onco_test = pd.read_csv(os.path.join(dir_datos, "dataset_prueba_onco.csv"), low_memory=False)
    df_control_test = pd.read_csv(os.path.join(dir_datos, "dataset_prueba_control.csv"), low_memory=False)
    
    df_onco_sample = df_onco_test.copy()
    n_onco_total = len(df_onco_sample)
    n_control_needed = 200000 - n_onco_total 
    
    proporciones_control = df_control_test[target_name].value_counts(normalize=True)
    df_ctrl_sample = df_control_test.groupby(target_name, group_keys=False).apply(
        lambda x: x.sample(min(len(x), int(np.round(n_control_needed * proporciones_control[x.name]))), random_state=42)
    )
    
    if len(df_ctrl_sample) != n_control_needed:
        if len(df_ctrl_sample) < n_control_needed:
            dif = n_control_needed - len(df_ctrl_sample)
            extras = df_control_test.drop(df_ctrl_sample.index).sample(n=dif, random_state=42)
            df_ctrl_sample = pd.concat([df_ctrl_sample, extras])
        elif len(df_ctrl_sample) > n_control_needed:
            dif = len(df_ctrl_sample) - n_control_needed
            df_ctrl_sample = df_ctrl_sample.drop(df_ctrl_sample.sample(n=dif, random_state=42).index)
            
    df_test_global = pd.concat([df_onco_sample, df_ctrl_sample], ignore_index=True).sample(frac=1, random_state=42)
    
    del df_onco_test, df_control_test, df_onco_sample, df_ctrl_sample
    gc.collect() 

    # Eliminar únicamente columnas objetivo que existan
    cols_excluir = [
        "MORTALIDAD",
        "SEVERIDAD",
        "CONSUMO_RECURSOS",
        "CATEGORIA_CANCER"
    ]

    cols_excluir = [
        c for c in cols_excluir
        if c in df_test_global.columns
    ]

    X_shap = df_test_global.drop(
        columns=cols_excluir,
        errors="ignore"
    ).astype("float32")

    del df_test_global
    gc.collect()

    # NOTA: Omitimos el bloque de filtrado de variables constantes (varianzas == 0)
    # ya que en el código original, la cohorte global NO eliminó la variable 
    # 'CATEGORIA_CANCER_SIN_CANCER' ni otras constantes para mantener compatibilidad con el modelo.

    # ======================================================================
    # CARGAR MATRIZ SHAP
    # ======================================================================

    print("Cargando backup SHAP global...")

    matriz_shap = np.load(ruta_backup)

    print(f"Matriz SHAP : {matriz_shap.shape}")
    print(f"Dataset      : {X_shap.shape}")

    if matriz_shap.shape[1] != X_shap.shape[1] or matriz_shap.shape[0] != X_shap.shape[0]:
        print("\nERROR")
        print("Las dimensiones no coinciden.")
        print(f"Matriz SHAP : {matriz_shap.shape[0]} filas, {matriz_shap.shape[1]} variables")
        print(f"Dataset     : {X_shap.shape[0]} filas, {X_shap.shape[1]} variables")

        raise ValueError(
            "La matriz SHAP y el dataset no coinciden en dimensiones."
        )

    # Clase 1 = Fallecido
    valores_shap = matriz_shap[:, :, 1]

    # ======================================================================
    # GENERAR DEPENDENCE PLOTS
    # ======================================================================

    print("\nGenerando dependence plots...\n")

    generados = 0

    for variable in variables_objetivo:

        if variable not in X_shap.columns:
            print(f"Advertencia: {variable} no existe en el dataset. Saltando...")
            continue

        print(f"Generando: {variable}")

        fig, ax = plt.subplots(figsize=(6, 4.5))

        shap.dependence_plot(
            variable,
            valores_shap,
            X_shap,
            interaction_index=None,
            ax=ax,
            show=False
        )

        ax.set_title(
            "Impacto en Riesgo de Fallecimiento (Clase 1)",
            fontsize=10
        )

        fig.suptitle(
            f"Dependence Plot: {variable} (Global)",
            fontsize=11,
            y=1.02
        )

        plt.tight_layout()

        # Guardar con el prefijo "EXTRA_" según lo solicitado
        plt.savefig(
            os.path.join(
                dir_salida,
                f"EXTRA_SHAP_Dependence_{variable}.png"
            ),
            dpi=250,
            bbox_inches="tight"
        )

        plt.close()

        generados += 1

    del matriz_shap
    del X_shap

    gc.collect()

    print("\n==============================================")
    print("Proceso terminado.")
    print(f"Dependence plots extra generados: {generados}")
    print(f"Carpeta de salida:\n{dir_salida}")
    print("==============================================")


# ==========================================================================
# EJECUTAR
# ==========================================================================

generar_dependence_plots_globales()

Reconstruyendo dataset global (Onco + Control)...


C:\Users\Carloto\AppData\Local\Temp\ipykernel_18408\1823941177.py:59: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_ctrl_sample = df_control_test.groupby(target_name, group_keys=False).apply(


Cargando backup SHAP global...
Matriz SHAP : (200000, 110, 2)
Dataset      : (200000, 110)

Generando dependence plots...

Generando: TIPO_DIAGNOSTICO_ONCO_SECUNDARIO
Generando: COMORBILIDAD_PRINCIPAL_SIN_COMORBILIDAD
Generando: TIPO_PROCEDIMIENTO_SISTEMA_DIGESTIVO

Proceso terminado.
Dependence plots extra generados: 3
Carpeta de salida:
../../Resultados/Resultados (etapa 5)/SHAP_MORTALIDAD/Valores SHAP (global)\Dependence_Plots
